# ETF Risk Model Validation

This notebook evaluates the ETF risk-scoring model through ranking checks, sensitivity analysis and comparison with realised downside outcomes.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

scoring_data = pd.read_csv(
    "../data/processed/etf_scoring_components.csv"
)

risk_ranking = pd.read_csv(
    "../data/processed/etf_risk_ranking.csv"
)

prices = pd.read_csv(
    "../data/processed/etf_prices_clean.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Scoring data:", scoring_data.shape)
print("Risk ranking:", risk_ranking.shape)
print("Price data:", prices.shape)

Scoring data: (20, 43)
Risk ranking: (20, 11)
Price data: (1284, 20)


In [2]:
split_date = pd.Timestamp("2024-07-22")

training_prices = prices.loc[prices.index <= split_date].copy()
testing_prices = prices.loc[prices.index > split_date].copy()

print("Full period:")
print(prices.index.min(), "to", prices.index.max())

print("\nTraining period:")
print(training_prices.index.min(), "to", training_prices.index.max())
print("Rows:", training_prices.shape[0])

print("\nTesting period:")
print(testing_prices.index.min(), "to", testing_prices.index.max())
print("Rows:", testing_prices.shape[0])

Full period:
2021-07-22 00:00:00 to 2026-07-22 00:00:00

Training period:
2021-07-22 00:00:00 to 2024-07-22 00:00:00
Rows: 772

Testing period:
2024-07-23 00:00:00 to 2026-07-22 00:00:00
Rows: 512


In [3]:
TRADING_DAYS = 252
benchmark_ticker = "VWRP.L"

training_returns = (
    training_prices
    .pct_change()
    .dropna(how="all")
)

# Annualised volatility
training_volatility = (
    training_returns.std()
    * np.sqrt(TRADING_DAYS)
)

# Maximum drawdown
training_growth = (1 + training_returns).cumprod()
training_peak = training_growth.cummax()

training_drawdown = (
    training_growth / training_peak
) - 1

training_max_drawdown = (
    training_drawdown.min().abs()
)

# 95% historical CVaR
training_cvar = training_returns.apply(
    lambda returns: abs(
        returns[
            returns <= returns.quantile(0.05)
        ].mean()
    )
)

# Beta relative to VWRP
benchmark_returns = training_returns[benchmark_ticker]
benchmark_variance = benchmark_returns.var()

training_beta = training_returns.apply(
    lambda returns:
        returns.cov(benchmark_returns)
        / benchmark_variance
).clip(lower=0)

training_metrics = pd.DataFrame({
    "Training_Volatility": training_volatility,
    "Training_Max_Drawdown": training_max_drawdown,
    "Training_CVaR": training_cvar,
    "Training_Beta": training_beta
})

training_metrics.index.name = "Ticker"

training_metrics.head()

,Training_Volatility,Training_Max_Drawdown,Training_CVaR,Training_Beta
Ticker,,,,
VWRP.L,0.128928,0.149201,0.018578,1.000000
SWDA.L,0.133747,0.153546,0.018989,1.024135
VUAG.L,0.142921,0.155470,0.020879,1.060739
ISF.L,0.128709,0.115010,0.020177,0.706890
EQQQ.L,0.195869,0.281650,0.027317,1.353087


In [4]:
def min_max_score(series):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(50, index=series.index)

    return 100 * (series - minimum) / (maximum - minimum)


training_scores = training_metrics.copy()

training_scores["Volatility_Score"] = min_max_score(
    training_scores["Training_Volatility"]
)

training_scores["Drawdown_Score"] = min_max_score(
    training_scores["Training_Max_Drawdown"]
)

training_scores["CVaR_Score"] = min_max_score(
    training_scores["Training_CVaR"]
)

training_scores["Beta_Score"] = min_max_score(
    training_scores["Training_Beta"]
)

training_scores["Training_Market_Risk_Score"] = (
    training_scores["Volatility_Score"] * 0.30
    + training_scores["Drawdown_Score"] * 0.30
    + training_scores["CVaR_Score"] * 0.25
    + training_scores["Beta_Score"] * 0.15
)

training_scores[
    [
        "Volatility_Score",
        "Drawdown_Score",
        "CVaR_Score",
        "Beta_Score",
        "Training_Market_Risk_Score"
    ]
].sort_values(
    "Training_Market_Risk_Score",
    ascending=False
).round(2)

,Volatility_Score,Drawdown_Score,CVaR_Score,Beta_Score,Training_Market_Risk_Score
Ticker,,,,,
ICHN.AS,100.00,100.00,100.00,73.74,96.06
INRG.L,84.21,92.14,93.16,82.60,88.59
IUIT.L,75.12,60.56,80.62,100.00,75.86
EQQQ.L,60.95,48.82,68.11,96.05,64.37
IWDP.L,44.06,53.49,49.43,56.29,50.07
VFEG.L,46.13,28.67,52.32,57.01,44.07
VUAG.L,42.36,20.86,50.09,75.30,42.78
VGOV.L,39.28,69.24,39.81,0.00,42.51
VEUR.L,42.29,22.70,49.46,64.12,41.48


In [5]:
# Include the final training price so the first testing return is calculated correctly
testing_prices_with_start = pd.concat([
    training_prices.tail(1),
    testing_prices
])

testing_returns = (
    testing_prices_with_start
    .pct_change()
    .dropna(how="all")
)

# Realised annualised volatility
testing_volatility = (
    testing_returns.std()
    * np.sqrt(TRADING_DAYS)
)

# Realised maximum drawdown
testing_growth = (1 + testing_returns).cumprod()
testing_peak = testing_growth.cummax()

testing_drawdowns = (
    testing_growth / testing_peak
) - 1

testing_max_drawdown = (
    testing_drawdowns.min().abs()
)

# Realised 95% CVaR
testing_cvar = testing_returns.apply(
    lambda returns: abs(
        returns[
            returns <= returns.quantile(0.05)
        ].mean()
    )
)

testing_outcomes = pd.DataFrame({
    "Testing_Volatility": testing_volatility,
    "Testing_Max_Drawdown": testing_max_drawdown,
    "Testing_CVaR": testing_cvar
})

testing_outcomes.index.name = "Ticker"

testing_outcomes.sort_values(
    "Testing_Max_Drawdown",
    ascending=False
).round(4)

,Testing_Volatility,Testing_Max_Drawdown,Testing_CVaR
Ticker,,,
INRG.L,0.3698,0.2884,0.0413
SWDA.L,0.3221,0.2742,0.0288
IUIT.L,0.2399,0.2640,0.0342
SGLN.L,0.2069,0.2489,0.0326
EQQQ.L,0.1885,0.2416,0.0276
ICHN.AS,0.2643,0.2285,0.0367
VUAG.L,0.1424,0.2088,0.0217
VWRP.L,0.1283,0.1764,0.0195
IUHC.L,0.1557,0.1763,0.0223


In [6]:
validation_results = (
    training_scores[
        ["Training_Market_Risk_Score"]
    ]
    .join(testing_outcomes, how="inner")
)

correlation_results = pd.DataFrame({
    "Pearson_Correlation": [
        validation_results["Training_Market_Risk_Score"].corr(
            validation_results["Testing_Volatility"],
            method="pearson"
        ),
        validation_results["Training_Market_Risk_Score"].corr(
            validation_results["Testing_Max_Drawdown"],
            method="pearson"
        ),
        validation_results["Training_Market_Risk_Score"].corr(
            validation_results["Testing_CVaR"],
            method="pearson"
        )
    ],
    "Spearman_Correlation": [
        validation_results["Training_Market_Risk_Score"].corr(
            validation_results["Testing_Volatility"],
            method="spearman"
        ),
        validation_results["Training_Market_Risk_Score"].corr(
            validation_results["Testing_Max_Drawdown"],
            method="spearman"
        ),
        validation_results["Training_Market_Risk_Score"].corr(
            validation_results["Testing_CVaR"],
            method="spearman"
        )
    ]
}, index=[
    "Future Volatility",
    "Future Maximum Drawdown",
    "Future CVaR"
])

correlation_results.round(3)

,Pearson_Correlation,Spearman_Correlation
Future Volatility,0.757,0.689
Future Maximum Drawdown,0.734,0.687
Future CVaR,0.807,0.696


In [7]:
from scipy.stats import pearsonr, spearmanr

significance_results = []

outcome_columns = {
    "Future Volatility": "Testing_Volatility",
    "Future Maximum Drawdown": "Testing_Max_Drawdown",
    "Future CVaR": "Testing_CVaR"
}

for outcome_name, outcome_column in outcome_columns.items():
    pearson_r, pearson_p = pearsonr(
        validation_results["Training_Market_Risk_Score"],
        validation_results[outcome_column]
    )

    spearman_r, spearman_p = spearmanr(
        validation_results["Training_Market_Risk_Score"],
        validation_results[outcome_column]
    )

    significance_results.append({
        "Outcome": outcome_name,
        "Pearson_Correlation": pearson_r,
        "Pearson_P_Value": pearson_p,
        "Spearman_Correlation": spearman_r,
        "Spearman_P_Value": spearman_p
    })

significance_results = pd.DataFrame(
    significance_results
).set_index("Outcome")

significance_results.round(4)

,Pearson_Correlation,Pearson_P_Value,Spearman_Correlation,Spearman_P_Value
Outcome,,,,
Future Volatility,0.7575,0.0001,0.6887,0.0008
Future Maximum Drawdown,0.7342,0.0002,0.6872,0.0008
Future CVaR,0.8068,0.0000,0.6962,0.0006


In [8]:
weight_scenarios = {
    "Base_Model": {
        "Volatility_Score": 0.30,
        "Drawdown_Score": 0.30,
        "CVaR_Score": 0.25,
        "Beta_Score": 0.15
    },
    "Volatility_Focused": {
        "Volatility_Score": 0.45,
        "Drawdown_Score": 0.20,
        "CVaR_Score": 0.20,
        "Beta_Score": 0.15
    },
    "Downside_Focused": {
        "Volatility_Score": 0.15,
        "Drawdown_Score": 0.40,
        "CVaR_Score": 0.35,
        "Beta_Score": 0.10
    },
    "Equal_Weights": {
        "Volatility_Score": 0.25,
        "Drawdown_Score": 0.25,
        "CVaR_Score": 0.25,
        "Beta_Score": 0.25
    }
}

sensitivity_scores = pd.DataFrame(
    index=training_scores.index
)

for scenario_name, weights in weight_scenarios.items():
    sensitivity_scores[scenario_name] = sum(
        training_scores[column] * weight
        for column, weight in weights.items()
    )

sensitivity_rankings = sensitivity_scores.rank(
    ascending=False,
    method="min"
)

sensitivity_rankings.head()

,Base_Model,Volatility_Focused,Downside_Focused,Equal_Weights
Ticker,,,,
VWRP.L,12.0,11.0,13.0,10.0
SWDA.L,10.0,9.0,12.0,9.0
VUAG.L,7.0,7.0,8.0,6.0
ISF.L,14.0,14.0,14.0,14.0
EQQQ.L,4.0,4.0,4.0,4.0


In [9]:
# Spearman rank correlation between each weighting scenario
rank_correlation = sensitivity_rankings.corr(
    method="spearman"
)

print("Rank correlations:")
display(rank_correlation.round(3))


# Rank movement relative to the base model
rank_changes = sensitivity_rankings.subtract(
    sensitivity_rankings["Base_Model"],
    axis=0
)

scenario_stability = pd.DataFrame({
    "Mean_Absolute_Rank_Change": (
        rank_changes.abs().mean()
    ),
    "Maximum_Absolute_Rank_Change": (
        rank_changes.abs().max()
    )
}).drop(index="Base_Model")

print("\nRank-change summary:")
display(scenario_stability.round(2))


# ETFs with the largest movement under any scenario
rank_changes["Largest_Absolute_Change"] = (
    rank_changes.drop(columns="Base_Model")
    .abs()
    .max(axis=1)
)

print("\nLargest ETF ranking movements:")
display(
    rank_changes.sort_values(
        "Largest_Absolute_Change",
        ascending=False
    ).head(10)
)

Rank correlations:


,Base_Model,Volatility_Focused,Downside_Focused,Equal_Weights
Base_Model,1.000,0.973,0.983,0.973
Volatility_Focused,0.973,1.000,0.938,0.995
Downside_Focused,0.983,0.938,1.000,0.938
Equal_Weights,0.973,0.995,0.938,1.000



Rank-change summary:


,Mean_Absolute_Rank_Change,Maximum_Absolute_Rank_Change
Volatility_Focused,0.7,5.0
Downside_Focused,0.6,3.0
Equal_Weights,0.7,5.0



Largest ETF ranking movements:


,Base_Model,Volatility_Focused,Downside_Focused,Equal_Weights,Largest_Absolute_Change
Ticker,,,,,
VGOV.L,0.0,5.0,-2.0,5.0,5.0
VJPN.L,0.0,-1.0,-3.0,-1.0,3.0
VWRP.L,0.0,-1.0,1.0,-2.0,2.0
SGLN.L,0.0,-2.0,1.0,-1.0,2.0
SWDA.L,0.0,-1.0,2.0,-1.0,2.0
AGGG.L,0.0,0.0,-1.0,0.0,1.0
VUAG.L,0.0,0.0,1.0,-1.0,1.0
VEUR.L,0.0,-1.0,0.0,-1.0,1.0
VFEG.L,0.0,0.0,1.0,1.0,1.0


In [10]:
validation_results.to_csv(
    "../data/processed/out_of_sample_validation.csv"
)

significance_results.to_csv(
    "../data/processed/validation_significance.csv"
)

sensitivity_scores.to_csv(
    "../data/processed/sensitivity_scores.csv"
)

sensitivity_rankings.to_csv(
    "../data/processed/sensitivity_rankings.csv"
)

rank_correlation.to_csv(
    "../data/processed/sensitivity_rank_correlations.csv"
)

scenario_stability.to_csv(
    "../data/processed/sensitivity_summary.csv"
)

rank_changes.to_csv(
    "../data/processed/sensitivity_rank_changes.csv"
)

print("All validation results saved successfully.")

All validation results saved successfully.
